# Pinch Analysis: Crude Preheat Train
This notebook shows how the pinch, utility targets, and graph shapes move when the minimum approach temperature assumption changes.


## Case context
The packaged `crude_preheat_train.json` case represents a small crude-unit preheat train with named hot and cold duties. The workflow is: solve the baseline case, inspect the main graphs, then rerun the case by changing the root-zone `dt_cont_multiplier`, which scales every process and utility stream `dt_cont` value through the zone hierarchy.


In [1]:
from copy import deepcopy
import json
import numpy as np
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from OpenPinch import PinchProblem

In [2]:
case_path = Path("/Users/timothyw/Github_Local/OpenPinch/examples/stream_data/p_Castillo.json")

In [3]:
problem = PinchProblem(source=case_path)
problem.target()
summary = problem.summary_frame()
base_row = summary.loc[
    summary["Target"] == "crude_preheat_train/Direct Integration",
    [
        "Hot Utility Target",
        "Cold Utility Target",
        "Heat Recovery",
        "Hot Pinch",
        "Cold Pinch",
    ],
]
print(summary)

                        Target  Hot Utility Target  Cold Utility Target  \
0      Site/Direct Integration                 0.0              885.419   
1    Site/Total Process Target                 0.0              885.419   
2       Site/Total Site Target                 0.0              885.419   
3  Castillo/Direct Integration                 0.0              885.419   

   Heat Recovery  Hot Pinch  Cold Pinch Hot Utilities        Cold Utilities  
0        3752.49      835.0       835.0     HPS: 0.00  CW: 885.42, CU: 0.00  
1        3752.49        NaN         NaN     HPS: 0.00  CW: 885.42, CU: 0.00  
2        3752.49       40.0        40.0     HPS: 0.00  CW: 885.42, CU: 0.00  
3        3752.49      835.0       835.0     HPS: 0.00  CW: 885.42, CU: 0.00  


## Baseline graphs
Render the baseline composite curve, shifted composite curve, and grand composite curve directly in the notebook so the graph shapes can be read before changing `dt_cont`.


In [4]:
problem.graph_catalog()

,Zone,Graph Type,Graph Name,Index
0,Direct Integration,Composite Curves,Graph 1,0
1,Direct Integration,Shifted Composite Curves,Graph 2,1
2,Direct Integration,Balanced Composite Curves,Graph 3,2
3,Direct Integration,Grand Composite Curve,Graph 4,3
4,Direct Integration,Net Load Profiles,Graph 5,4
5,Direct Integration,Grand Composite Curve with Heat Pump,Graph 6,5
6,Total Site Target,Total Site Profiles,Graph 1,0
7,Total Site Target,Site Utility Grand Composite Curve,Graph 2,1


In [5]:
cc = problem.plot.composite_curve(show=True)

In [6]:
scc = problem.plot.shifted_composite_curve(show=True)

In [7]:
bcc = problem.plot.balanced_composite_curve(show=True)

In [8]:
gcc = problem.plot.grand_composite_curve(show=True)

In [9]:
nlc = problem.plot.net_load_profiles(show=True)

In [10]:
base_payload = json.loads(case_path.read_text(encoding="utf-8"))
multipliers = np.linspace(0.5, 10.0, 26)


def _build_zone_tree_with_multiplier(payload: dict, factor: float) -> dict:
    zone_tree = payload.get("zone_tree")
    if not zone_tree:
        root_name = payload.get("name") or case_path.stem
        root = {"name": root_name, "type": "Site", "children": {}}

        for stream in payload.get("streams", []):
            zone_label = stream.get("zone")
            if not zone_label:
                continue
            current = root
            for part in [part.strip() for part in zone_label.split("/") if part.strip()]:
                current = current["children"].setdefault(
                    part,
                    {"name": part, "type": "Process Zone", "children": {}},
                )

        def _to_schema(node: dict) -> dict:
            children = [_to_schema(child) for child in node["children"].values()]
            result = {"name": node["name"], "type": node["type"]}
            if children:
                result["children"] = children
            return result

        zone_tree = _to_schema(root)
        payload["zone_tree"] = zone_tree
    zone_tree["dt_cont_multiplier"] = float(factor)
    return payload


rows = []
for factor in multipliers:
    payload = _build_zone_tree_with_multiplier(deepcopy(base_payload), factor)
    variant_problem = PinchProblem(project_name=case_path.stem, source=payload)
    variant_problem.target()
    variant_summary = variant_problem.summary_frame()
    row = variant_summary.loc[
        variant_summary["Target"] == f"{case_path.stem}/Direct Integration"
    ].iloc[0]
    rows.append(
        {
            "dt_cont multiplier": factor,
            "Hot Utility Target": row["Hot Utility Target"],
            "Cold Utility Target": row["Cold Utility Target"],
            "Heat Recovery": row["Heat Recovery"],
            "Hot Pinch": row["Hot Pinch"],
            "Cold Pinch": row["Cold Pinch"],
        }
    )

sensitivity = pd.DataFrame(rows)
sensitivity

,dt_cont multiplier,Hot Utility Target,Cold Utility Target,Heat Recovery,Hot Pinch,Cold Pinch
0,0.50,0.0,885.419,3752.49,837.5,837.5
1,0.88,0.0,885.419,3752.49,835.6,835.6
2,1.26,0.0,885.419,3752.49,833.7,833.7
3,1.64,0.0,885.419,3752.49,831.8,831.8
4,2.02,0.0,885.419,3752.49,829.9,829.9
5,2.40,0.0,885.419,3752.49,828.0,828.0
6,2.78,0.0,885.419,3752.49,826.1,826.1
7,3.16,0.0,885.419,3752.49,824.2,824.2
8,3.54,0.0,885.419,3752.49,822.3,822.3
9,3.92,0.0,885.419,3752.49,820.4,820.4


In [11]:
# base_payload = json.loads(case_path.read_text(encoding="utf-8"))
# multipliers = np.linspace(0.5, 10.0, 26)

# rows = []
# for factor in multipliers:
#     problem.master_zone.dt_cont_multiplier = factor
#     problem.target()
#     summary = problem.summary_frame()
#     row = summary.loc[
#         summary["Target"] == f"{case_path.stem}/Direct Integration"
#     ].iloc[0]
#     rows.append(
#         {
#             "dt_cont multiplier": factor,
#             "Hot Utility Target": row["Hot Utility Target"],
#             "Cold Utility Target": row["Cold Utility Target"],
#             "Heat Recovery": row["Heat Recovery"],
#             "Hot Pinch": row["Hot Pinch"],
#             "Cold Pinch": row["Cold Pinch"],
#         }
#     )

# sensitivity = pd.DataFrame(rows)
# sensitivity

In [12]:
series_colors = {
    "Hot Utility Target": "#c0392b",
    "Cold Utility Target": "#2471a3",
    "Heat Recovery": "#138d75",
    "Hot Pinch": "#d68910",
    "Cold Pinch": "#7d3c98",
}

sensitivity_fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Utility Targets and Heat Recovery",
        "Pinch Temperatures",
    ),
)

for column in ["Hot Utility Target", "Cold Utility Target", "Heat Recovery"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=1,
        col=1,
    )

for column in ["Hot Pinch", "Cold Pinch"]:
    sensitivity_fig.add_trace(
        go.Scatter(
            x=sensitivity["dt_cont multiplier"],
            y=sensitivity[column],
            mode="lines+markers",
            name=column,
            line={"color": series_colors[column], "width": 3},
            marker={"color": series_colors[column], "size": 7},
        ),
        row=2,
        col=1,
    )

sensitivity_fig.update_xaxes(title_text="dt_cont multiplier", row=2, col=1)
sensitivity_fig.update_yaxes(title_text="kW or degC", row=1, col=1)
sensitivity_fig.update_yaxes(title_text="degC", row=2, col=1)
sensitivity_fig.update_layout(title="dt_cont sensitivity overview")
# display_plotly(sensitivity_fig, height=700)